# 8장 — 눈금 갈아타기: 양자화 (핵심) 🗜️

웹 챕터의 두 실험(눈금 갈아타기 기계와 다음 토큰 대결)을 같은 toy 벡터로 재현하고,
"왜 추론은 버티나"의 세 겹 보험(상쇄·argmax·분포)을 실측합니다. 마지막엔 9장의 빌런
**아웃라이어**를 미리 소환해 봅니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: 4GB 파일의 첫 원리.
**"정수 q 로 저장하고, 축척 스티커 s 로 되살린다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch8')

## 1. 눈금 갈아타기 기계: quantize / dequantize 다섯 줄

In [ ]:
import numpy as np

def quantize(w, bits=8):
    """대칭 양자화: s = max|w|/눈금수, q = round(w/s). 반환 (정수 q, 축척 s)."""
    levels = 2 ** (bits - 1) - 1          # int8 → 127, int4 → 7, int2 → 1
    s = np.abs(w).max() / levels
    return np.round(w / s).astype(int), s

def dequantize(q, s):
    return q * s

w = np.array([0.11, -0.08, 0.05, 0.02, -0.03, 0.07, -0.11, 0.01])  # 웹 실험 ①과 동일
q, s = quantize(w, bits=8)
w_hat = dequantize(q, s)
print(f'축척 s = {s:.6f}   (= 0.11 / 127)')
print('정수 q  :', q)
print('복원    :', np.round(w_hat, 5))
print('오차    :', np.round(np.abs(w_hat - w), 6))

assert np.abs(w_hat - w).max() <= s / 2 + 1e-12, '오차 한도 = 눈금 절반 s/2 — 반올림의 보장'
assert q.min() >= -127 and q.max() <= 127, '저장물은 1바이트 정수'
print()
print('✔ 확인: 저장은 정수 q 들 + 축척 s 하나. 4바이트가 1바이트로 (7B: 26GB → 7GB)')

## 2. 다음 토큰 대결: 박빙(마진 0.005)은 몇 비트까지 버티나

웹과 같은 미니 출력층: fp 로짓 [0.295, 0.290, −0.275, 0.049], 고양이 vs 강아지.

In [ ]:
VOCAB = ['고양이', '강아지', '토끼', '거북이']
W = np.array([
    [0.11, -0.08, 0.05, 0.02, -0.03, 0.07, -0.11, 0.01],
    [0.08, -0.03, 0.09, 0.00, 0.01, 0.06, 0.07, 0.09],
    [-0.02, 0.10, -0.07, 0.01, 0.06, -0.05, 0.02, -0.04],
    [-0.10, 0.03, 0.08, -0.06, 0.04, 0.01, -0.03, 0.09],
])
x = np.array([0.8, -0.5, 1.2, 0.3, -0.9, 0.6, -0.2, 1.0])

fp = W @ x
print(f'fp 로짓: {np.round(fp, 3)} → 1등 {VOCAB[fp.argmax()]} (마진 {fp[0] - fp[1]:.3f})')

results = {}
for bits in [8, 4, 3, 2]:
    Wq = np.vstack([dequantize(*quantize(row, bits)) for row in W])
    lq = Wq @ x
    results[bits] = lq
    flag = '✅ 유지' if lq.argmax() == fp.argmax() else '❗ 역전'
    print(f'int{bits}: {np.round(lq, 3)} → 1등 {VOCAB[lq.argmax()]}  {flag}  (로짓 오차 {np.abs(lq - fp).max():.4f})')

assert results[8].argmax() == fp.argmax() and results[4].argmax() == fp.argmax(), 'int8·int4: 박빙도 지킨다'
assert results[3].argmax() != fp.argmax() and results[2].argmax() != fp.argmax(), 'int3 부터 역전 — 오차 > 마진'
assert np.abs(results[8] - fp).max() < 0.005 < np.abs(results[3] - fp).max(), '갈림길 = 오차와 마진의 대소'
print()
print('✔ 확인: 결정(argmax)은 로짓 오차 < 마진인 한 무사. 실전 마진은 대부분 이보다 크다')

## 3. 세 겹 보험 실측: 내적 상쇄 (4장의 √n, 세 번째 출근)

차원 4,096 짜리 진짜 크기의 내적에서, 실측 오차가 최악의 경우보다 얼마나 작은지.

In [ ]:
rng = np.random.default_rng(9)                      # 시드 고정 — 결정적
d = 4096
w_big = rng.normal(0, 0.02, d)                      # 0 중심 종모양 가중치 (3부 5장)
x_big = rng.normal(0, 1.0, d)

q_big, s_big = quantize(w_big, bits=8)
err_vec = dequantize(q_big, s_big) - w_big

measured = abs(float(err_vec @ x_big))              # 실측 내적 오차
worst = float((s_big / 2) * np.abs(x_big).sum())    # 최악: 모든 티끌이 같은 방향일 때
signal = abs(float(w_big @ x_big))

print(f'내적 참값 크기   : {signal:.4f}')
print(f'실측 오차        : {measured:.6f}')
print(f'최악의 경우 한도 : {worst:.4f}   (실측이 {worst / measured:,.0f}배 작다)')

assert measured < worst / 10, '± 티끌이 랜덤워크처럼 상쇄 — √4096 = 64배 급 이득 (4장)'
assert measured / signal < 0.01, '로짓 상대 오차 1% 미만 — argmax 가 버티는 이유'
print()
print('✔ 확인: 오차 작음(s/2) × 상쇄(√n) × argmax 둔감. 추론이 int8 을 버티는 세 겹 보험')

## 4. 분포의 축복, 그리고 빌런의 예고편: 아웃라이어

종모양 가중치에 **큰 값 하나**가 끼면 무슨 일이 생길까요? (9장의 문제의식을 미리 실측)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

clean = w_big                                        # 종모양 (max ≈ 0.08)
dirty = clean.copy(); dirty[0] = 0.5                 # 아웃라이어 한 개 침입!

_, s_clean = quantize(clean, 8)
_, s_dirty = quantize(dirty, 8)
err_clean = np.abs(dequantize(*quantize(clean, 8)) - clean).mean()
err_dirty = np.abs(dequantize(*quantize(dirty, 8)) - dirty)[1:].mean()   # 나머지 4095개의 평균 오차

print(f'축척: 깨끗한 분포 {s_clean:.2e} → 아웃라이어 침입 후 {s_dirty:.2e} ({s_dirty / s_clean:.1f}배 성겨짐)')
print(f'나머지 값들의 평균 오차: {err_clean:.2e} → {err_dirty:.2e} ({err_dirty / err_clean:.1f}배 악화)')

assert s_dirty > s_clean * 4, '단 한 개의 아웃라이어가 모두의 축척을 망친다'
assert err_dirty > err_clean * 4, '눈금이 성겨져 나머지 4,095개의 오차가 함께 커진다'

plt.figure(figsize=(6.5, 3.8))
plt.hist(clean, bins=80, color='tab:purple', alpha=0.7, label='가중치 분포 (0 중심 종모양)')
plt.axvline(0.5, color='tab:red', ls='--', label='아웃라이어 0.5: 축척을 4배 이상 망침')
plt.axvline(float(np.abs(clean).max()), color='tab:gray', ls=':', label=f'원래 max ≈ {np.abs(clean).max():.2f}')
plt.xlabel('가중치 값'); plt.ylabel('개수'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title('분포의 축복과 빌런: 큰 값 하나가 모두의 눈금을 성기게 한다'); plt.tight_layout(); plt.show()
print('✔ 확인: 9장 예고. 해법은 "몇십 개마다 축척 하나"(블록 양자화), q4_K_M 의 핵심')

**직접 해보기**: ① s2 의 x 를 조금 바꿔 마진을 키우면 int3 도 순위를 지키는지 확인해 보세요.
② s1 의 bits 를 4로 바꿔서 눈금 15개(±7)로도 오차가 s/2 안에 있는 건 그대로임을 확인해 보세요
(문제는 그 s 자체가 커진다는 것).

## 5. 관찰 정리 + 🤖 AI 연결

- **양자화** = 실수 자 → 정수 자 + 축척: s = max|w|/127, q = round(w/s), 복원 q×s.
  저장 = 정수(1바이트) + 묶음당 s. 7B 가 26GB → **7GB (Ollama q8_0)**. 오차 한도 s/2.
- **왜 버티나 (세 겹 보험)**: 오차 작음 × 내적 상쇄(√n, 실측 수십 배) × argmax 둔감
  (박빙 0.005 도 int8·int4 는 지켰고 int3 부터 역전).
- **분포의 축복**(3부 5장): 0 중심 종모양 → s 작음. **아웃라이어 한 개**가 축척을 4배 망치고
  나머지 전원의 오차를 키운다. 9장(블록 양자화·q4_K_M)의 문제의식.
- **다시 쓰는 수학**: 1부 2장(내적), 3부 5장(분포), 2장(int8·zero-point 의 비대칭), 4장(√n).

**다음 챕터**: 9장(4비트까지: 블록 양자화와 GGUF). 눈금 15개로 버티는 비결과
`Q4_K_M` 파일명 완독.